# 02 — Sınav Takvimi Problemi

**Öğrenme hedefleri:**
1. Binary (0/1) karar değişkenlerini kullanmak
2. "Her ders tam olarak bir slota" tipi atama kısıtları yazmak
3. Çakışma kısıtlarını modellemek
4. Graph coloring bağlantısını görmek
5. Objective fonksiyonu değiştirmenin etkisini gözlemlemek

**Tahmini süre:** ~35 dakika

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q pulp
    !git clone -q https://github.com/Meliksahcaliskan/ILP_Workshop.git
    sys.path.insert(0, "ILP_Workshop")

In [ ]:
import pulp
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import combinations

---
## Veri

Kurgusal bir bölümde 9 ders ve her dersin hangi öğrenciler tarafından alındığı bilgisi:  
(Deterministik — her çalıştırmada aynı veriyi üretir)

In [ ]:
courses = ["CSE101", "CSE211", "CSE301", "CSE315", "CSE401",
           "MATH201", "MATH301", "EE201", "EE301"]

# Her öğrencinin kayıtlı olduğu dersler (sabit, deterministik)
student_enrollments = [
    ["CSE101", "MATH201", "EE201"],
    ["CSE101", "CSE211", "MATH201"],
    ["CSE211", "CSE301", "EE201"],
    ["CSE101", "CSE315", "MATH301"],
    ["CSE211", "MATH201", "EE301"],
    ["CSE301", "CSE401", "MATH201"],
    ["CSE315", "MATH301", "EE201"],
    ["CSE101", "CSE401", "EE301"],
    ["CSE211", "CSE301", "MATH301"],
    ["CSE315", "EE201", "EE301"],
    ["CSE101", "MATH201", "CSE401"],
    ["CSE211", "MATH301", "EE201"],
    ["CSE301", "CSE315", "EE301"],
    ["CSE401", "MATH201", "EE201"],
    ["CSE101", "CSE211", "EE301"],
    ["MATH201", "MATH301", "EE301"],
    ["CSE301", "EE201", "MATH201"],
    ["CSE315", "CSE401", "MATH301"],
    ["CSE101", "CSE301", "EE201"],
    ["CSE211", "CSE315", "MATH201"],
]

# Ders başına öğrenci sayısı
enrollment_count = {c: sum(c in s for s in student_enrollments) for c in courses}

df = pd.DataFrame({
    "Ders": courses,
    "Öğrenci Sayısı": [enrollment_count[c] for c in courses]
})
print(df.to_string(index=False))
print(f"\nToplam öğrenci: {len(student_enrollments)}")

---
## Çakışma Grafı

İki ders arasında ortak öğrenci varsa, bu dersler **çakışıyor** demektir — aynı sınav slotuna konulamazlar.

In [ ]:
# Çakışan ders çiftlerini bul
conflict_pairs = set()
for enrollment in student_enrollments:
    for c1, c2 in combinations(enrollment, 2):
        conflict_pairs.add(tuple(sorted([c1, c2])))

print(f"Toplam çakışan ders çifti: {len(conflict_pairs)}")

# Graf oluştur ve çiz
G = nx.Graph()
G.add_nodes_from(courses)
G.add_edges_from(conflict_pairs)

pos = nx.spring_layout(G, seed=42)
fig, ax = plt.subplots(figsize=(9, 6))
nx.draw_networkx(
    G, pos=pos, ax=ax,
    node_color="steelblue", node_size=700,
    font_color="white", font_size=8, font_weight="bold",
    edge_color="gray", alpha=0.9
)
ax.set_title("Çakışma Grafı — Düğüm: Ders, Kenar: Ortak Öğrenci")
ax.axis("off")
plt.tight_layout()
plt.show()

print()
print("Bu bir graph coloring problemi: 'renk = sınav slotu'")
print("Bağlı düğümler farklı renk almalı → çakışan dersler farklı slota gitmalı.")
print()
print("İlginç bağlantı: Derleyicilerde register allocation problemi de aynı yapıya sahip!")

---
## ILP Modeli

**Karar değişkenleri:**
- `x[c, t] ∈ {0, 1}` — ders `c`, slot `t`'de ise 1, değilse 0
- `y[t] ∈ {0, 1}` — slot `t` kullanılıyorsa 1

**Amaç:** Kullanılan slot sayısını minimize et: `min Σ_t y[t]`

**Kısıtlar:**
1. Her ders tam olarak bir slota: `Σ_t x[c,t] = 1`
2. Çakışan dersler aynı slotta olamaz: `x[c1,t] + x[c2,t] ≤ 1`
3. Slot bağlantısı: `x[c,t] ≤ y[t]`

In [ ]:
MAX_SLOTS = len(courses)  # üst sınır
slots = list(range(MAX_SLOTS))

prob = pulp.LpProblem("exam_scheduling", pulp.LpMinimize)

x = {(c, t): pulp.LpVariable(f"x_{c}_{t}", cat="Binary")
     for c in courses for t in slots}
y = {t: pulp.LpVariable(f"y_{t}", cat="Binary") for t in slots}

# Objective
prob += pulp.lpSum(y[t] for t in slots)

# Her ders tam olarak bir slota
for c in courses:
    prob += pulp.lpSum(x[c, t] for t in slots) == 1

# Çakışan dersler aynı slotta olamaz
for (c1, c2) in conflict_pairs:
    for t in slots:
        prob += x[c1, t] + x[c2, t] <= 1

# Slot kullanım bağlantısı
for c in courses:
    for t in slots:
        prob += x[c, t] <= y[t]

prob.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Durum           : {pulp.LpStatus[prob.status]}")
print(f"Kullanılan slot : {int(pulp.value(prob.objective))}")

---
## Sonuç Görselleştirmesi

In [ ]:
# Hangi ders hangi slotta?
schedule = {}
for c in courses:
    for t in slots:
        if pulp.value(x[c, t]) > 0.5:
            schedule[c] = t

# Takvim tablosu
print("=== Sınav Takvimi ===")
used_slots = sorted(set(schedule.values()))
for t in used_slots:
    courses_in_slot = [c for c, s in schedule.items() if s == t]
    print(f"  Slot {t+1}: {', '.join(courses_in_slot)}")

# Renklendirilmiş çakışma grafı
color_map_palette = plt.cm.Set2.colors
node_colors = [color_map_palette[schedule[c] % len(color_map_palette)] for c in G.nodes()]

fig, ax = plt.subplots(figsize=(9, 6))
nx.draw_networkx(
    G, pos=pos, ax=ax,
    node_color=node_colors, node_size=700,
    font_color="white", font_size=8, font_weight="bold",
    edge_color="gray", alpha=0.9
)
legend_handles = [
    mpatches.Patch(color=color_map_palette[t % len(color_map_palette)], label=f"Slot {t+1}")
    for t in used_slots
]
ax.legend(handles=legend_handles, loc="upper right")
ax.set_title(f"Renklendirilmiş Çakışma Grafı — {len(used_slots)} Slot")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 🔧 Değiştir-Gözlemle Görevleri

### Görev 1 — Hoca Kısıtı

> **Tahmin:** CSE101'in Slot 0'da yapılamayacağını eklersek slot sayısı değişir mi?

In [ ]:
prob_g1 = pulp.LpProblem("exam_g1", pulp.LpMinimize)

x1 = {(c, t): pulp.LpVariable(f"x_{c}_{t}", cat="Binary")
      for c in courses for t in slots}
y1 = {t: pulp.LpVariable(f"y_{t}", cat="Binary") for t in slots}

prob_g1 += pulp.lpSum(y1[t] for t in slots)
for c in courses:
    prob_g1 += pulp.lpSum(x1[c, t] for t in slots) == 1
for (c1, c2) in conflict_pairs:
    for t in slots:
        prob_g1 += x1[c1, t] + x1[c2, t] <= 1
for c in courses:
    for t in slots:
        prob_g1 += x1[c, t] <= y1[t]

# YENİ KISIT: CSE101 slot 0'da olamaz
prob_g1 += x1["CSE101", 0] == 0

prob_g1.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Durum           : {pulp.LpStatus[prob_g1.status]}")
print(f"Kullanılan slot : {int(pulp.value(prob_g1.objective))}")

> **Eğitmen notu:** Genellikle slot sayısı değişmez; solver aynı ders planını farklı bir slot permütasyonuyla bulur. Ancak kısıt bağlama etkisine bağlı olarak 1 artabilir.

### Görev 2 — Salon Kapasitesi

> **Tahmin:** Her slot'ta toplam öğrenci sayısı 25 ile sınırlı olursa slot sayısı nasıl değişir?

In [ ]:
CAPACITY = 25  # Değiştirerek dene: 20, 15, 10

prob_g2 = pulp.LpProblem("exam_g2", pulp.LpMinimize)
x2 = {(c, t): pulp.LpVariable(f"x_{c}_{t}", cat="Binary")
      for c in courses for t in slots}
y2 = {t: pulp.LpVariable(f"y_{t}", cat="Binary") for t in slots}

prob_g2 += pulp.lpSum(y2[t] for t in slots)
for c in courses:
    prob_g2 += pulp.lpSum(x2[c, t] for t in slots) == 1
for (c1, c2) in conflict_pairs:
    for t in slots:
        prob_g2 += x2[c1, t] + x2[c2, t] <= 1
for c in courses:
    for t in slots:
        prob_g2 += x2[c, t] <= y2[t]

# KAPASİTE KISITI
for t in slots:
    prob_g2 += pulp.lpSum(enrollment_count[c] * x2[c, t] for c in courses) <= CAPACITY

prob_g2.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Kapasite        : {CAPACITY}")
print(f"Durum           : {pulp.LpStatus[prob_g2.status]}")
if prob_g2.status == 1:
    print(f"Kullanılan slot : {int(pulp.value(prob_g2.objective))}")

> **Eğitmen notu:** Kapasiteyi düşürdükçe gereken slot sayısı artar. Yeterince küçük yapıldığında (ör. 5) model **Infeasible** olur.

### Görev 3 — Optimumdan 1 Eksik Slot

> **Tahmin:** Bulunan optimal slot sayısını 1 azaltırsak ne olur?  
> (Graf teorisinde bu, "kromatik sayıdan az renkle boyama" denemesine eşdeğer)

In [ ]:
optimal_slots = int(pulp.value(prob.objective))
restricted_slots = list(range(optimal_slots - 1))  # 1 eksik

prob_g3 = pulp.LpProblem("exam_g3", pulp.LpMinimize)
x3 = {(c, t): pulp.LpVariable(f"x_{c}_{t}", cat="Binary")
      for c in courses for t in restricted_slots}
y3 = {t: pulp.LpVariable(f"y_{t}", cat="Binary") for t in restricted_slots}

prob_g3 += pulp.lpSum(y3[t] for t in restricted_slots)
for c in courses:
    prob_g3 += pulp.lpSum(x3[c, t] for t in restricted_slots) == 1
for (c1, c2) in conflict_pairs:
    for t in restricted_slots:
        prob_g3 += x3[c1, t] + x3[c2, t] <= 1
for c in courses:
    for t in restricted_slots:
        prob_g3 += x3[c, t] <= y3[t]

prob_g3.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Optimal slot sayısı : {optimal_slots}")
print(f"Denenen slot sayısı : {optimal_slots - 1}")
print(f"Durum               : {pulp.LpStatus[prob_g3.status]}")

> **Eğitmen notu:** Durum **Infeasible**. Grafın kromatik sayısından daha az renkle boyama yapmak mümkün değil.

### Görev 4 — Objective Değiştir: Art Arda Sınav (İleri Seviye — Eğitmen Gösterir)

Slot sayısını sabit tutup "art arda iki sınavı olan öğrenci sayısını" minimize etmeye çalışalım.  
Bunun için yeni binary yardımcı değişkenler gerekiyor.  

> Bu görev **yeni decision variable ekleme** örneği olarak kullanılır.

In [ ]:
# Slot sayısını optimal değerde sabitle
fixed_slots = list(range(optimal_slots))

prob_g4 = pulp.LpProblem("exam_g4_consecutive", pulp.LpMinimize)

x4 = {(c, t): pulp.LpVariable(f"x_{c}_{t}", cat="Binary")
      for c in courses for t in fixed_slots}

# z[s, t1, t2] = öğrenci s'nin t1 ve t2'de (art arda) sınavı var mı
consec_slots = [(t, t+1) for t in fixed_slots[:-1]]
z = {(sid, t1, t2): pulp.LpVariable(f"z_{sid}_{t1}_{t2}", cat="Binary")
     for sid, enrollment in enumerate(student_enrollments)
     for (t1, t2) in consec_slots
     if len(enrollment) >= 2}

# Objective: art arda sınav olan (öğrenci, slot çifti) sayısını minimize et
prob_g4 += pulp.lpSum(z.values())

# Her ders tam olarak bir slota
for c in courses:
    prob_g4 += pulp.lpSum(x4[c, t] for t in fixed_slots) == 1

# Çakışma kısıtları
for (c1, c2) in conflict_pairs:
    for t in fixed_slots:
        prob_g4 += x4[c1, t] + x4[c2, t] <= 1

# z değişkenlerini x'e bağla
for sid, enrollment in enumerate(student_enrollments):
    for (t1, t2) in consec_slots:
        # z = 1 ancak öğrenci hem t1'de hem t2'de sınavı varsa
        for c1 in enrollment:
            for c2 in enrollment:
                if c1 != c2:
                    prob_g4 += z[sid, t1, t2] >= x4[c1, t1] + x4[c2, t2] - 1

prob_g4.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Durum                              : {pulp.LpStatus[prob_g4.status]}")
print(f"Art arda sınav olan (öğrenci,slot) : {int(pulp.value(prob_g4.objective))}")